# KinetiDiff MD Validation — Orchestration Notebook

**Purpose:** Thin orchestration layer. All computation lives in
`src/kinetidiff/molecular_dynamics/`. This notebook:
1. Loads the config and displays the campaign matrix.
2. Shows SLURM job submission commands and monitoring snippets (run those on ORCD).
3. Loads completed result CSVs from ORCD scratch and renders publication figures.
4. Calls `postprocess.strip_and_stride` and `sync_to_drive_bundle.sh` to build the Drive bundle.

**No business logic belongs in cells. If you are writing a computation loop, move it to a module.**

---
**Campaign:** 5 leads × 2 receptors (WT + R206H) × 3 replicas × 100 ns = **30 runs ≈ 3,000 ns**  
**Platform:** MIT ORCD Engaging — `mit_preemptable` (GPU) + `mit_normal` (CPU analysis)  
**Drive sync:** Manual (bundle < 100 GB, water-stripped + strided)

## Cell 0 — Environment Setup

In [ ]:
import subprocess, sys
from pathlib import Path

# Locate repo root robustly
try:
    REPO_ROOT = Path(
        subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
    )
except Exception:
    # Fallback if not in a git repo context
    REPO_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent

# Add src/ to sys.path so kinetidiff is importable without pip install -e
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Python:    {sys.executable} ({sys.version.split()[0]})')

## Cell 1 — Load Config & Campaign Matrix

In [ ]:
from omegaconf import OmegaConf
import pandas as pd

CONFIG_PATH = REPO_ROOT / 'configs' / 'simulation.yaml'
cfg = OmegaConf.load(CONFIG_PATH)
cfg_dict = OmegaConf.to_container(cfg, resolve=True)

from kinetidiff.molecular_dynamics.simulation.runner import build_run_matrix
matrix = build_run_matrix(cfg_dict['campaign'])

matrix_df = pd.DataFrame(matrix, columns=['ligand_id', 'receptor', 'replica_id', 'seed'])
matrix_df.index.name = 'slurm_task_id'
print(f'Campaign matrix: {len(matrix_df)} runs')
display(matrix_df.head(10))

sim = cfg_dict['simulation']
ns_per_run = sim['production_steps'] * sim['timestep_ps'] * 1e-3
print(f'\nProduction length: {ns_per_run:.0f} ns/run × {len(matrix_df)} runs = {ns_per_run * len(matrix_df):.0f} ns total')

## Cell 2 — ORCD Submission Commands

Run these **on the ORCD Engaging login node** (not in this notebook).
Copy-paste into your terminal after `ssh engaging.mit.edu`.

In [ ]:
print("""
# ── Step 1: Build conda environment (once) ────────────────────────────────────
module purge && module load miniforge
conda env create -f environment-md.yml
conda activate kinetidiff-md

# ── Step 2: Quick smoke test (10 ps, mit_quicktest, ~5 min) ──────────────────
sbatch --partition=mit_quicktest -c 2 --mem=8G -t 00:15:00 \\
    --wrap="module purge && module load miniforge && conda activate kinetidiff-md && \\
    python -m kinetidiff.molecular_dynamics.simulation.runner \\
    --config configs/simulation.yaml --repo-root $(pwd) --smoke-test"

# ── Step 3: Submit equilibration array (10 systems, mit_normal_gpu, 6 h) ────
EQUIL_JOB=$(sbatch scripts/md/submit_equil_array.sh | awk '{print $NF}')
echo "Equilibration job ID: ${EQUIL_JOB}"

# ── Step 4: Submit production array (30 runs, mit_preemptable, 48 h, --requeue)
PROD_JOB=$(sbatch --dependency=afterok:${EQUIL_JOB} scripts/md/submit_prod_array.sh | awk '{print $NF}')
echo "Production job ID: ${PROD_JOB}"

# ── Step 5: Submit analysis array (CPU, 4 h, depends on production) ─────────
ANA_JOB=$(sbatch --dependency=afterok:${PROD_JOB} scripts/md/submit_analysis_array.sh | awk '{print $NF}')
echo "Analysis job ID: ${ANA_JOB}"

# ── Monitoring ────────────────────────────────────────────────────────────────
squeue --me
sinfo -p mit_preemptable -O Partition,Nodes,CPUsState,Gres:30,GresUsed:30,StateCompact -e
# tail -f logs/md/prod_<JOBID>_0.log
""")

## Cell 3 — Load Completed Results

Run this cell **after all analysis tasks have completed** on ORCD.  
Set `SCRATCH_DIR` to your local path if pulling results via `scp`.

In [ ]:
import os
import json

SCRATCH_DIR = Path(os.environ.get('ORCD_SCRATCH', Path.home() / 'orcd' / 'scratch')) / 'kinetidiff-md'
ANALYSIS_DIR = SCRATCH_DIR / 'analysis'
FIGURES_DIR  = SCRATCH_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Looking for results in: {ANALYSIS_DIR}')
all_tags = [p.stem for p in ANALYSIS_DIR.glob('*') if p.is_dir()]
print(f'Completed runs found: {len(all_tags)}')
for t in sorted(all_tags):
    print(f'  {t}')

In [ ]:
from kinetidiff.molecular_dynamics.simulation.runner import build_run_matrix

# Collect per-run analysis data
ANALYSIS_RESULTS = {}  # (lig_id, receptor, replica_id) -> {rmsd, rmsf, hbonds, pocket}
MMGBSA_RESULTS   = {}  # (lig_id, receptor, replica_id) -> mmgbsa dict

for lig_id, receptor, replica_id, seed in matrix:
    tag = f'{lig_id}_{receptor}_rep{replica_id}'
    run_dir = ANALYSIS_DIR / tag
    key = (lig_id, receptor, replica_id)

    res = {}
    for metric in ('rmsd', 'rmsf', 'hbonds'):
        csv_path = run_dir / f'{metric}.csv'
        if csv_path.exists():
            res[metric] = pd.read_csv(csv_path)
        else:
            res[metric] = pd.DataFrame()

    pocket_path = run_dir / 'pocket_contacts.json'
    res['pocket'] = json.loads(pocket_path.read_text()) if pocket_path.exists() else {}
    ANALYSIS_RESULTS[key] = res

    mmgbsa_csv = run_dir / 'mmgbsa.csv'
    if mmgbsa_csv.exists():
        row = pd.read_csv(mmgbsa_csv).iloc[0].to_dict()
        MMGBSA_RESULTS[key] = row

print(f'Loaded analysis results for {sum(1 for r in ANALYSIS_RESULTS.values() if not r["rmsd"].empty)} / {len(matrix)} runs.')
print(f'MM-GBSA results available: {len(MMGBSA_RESULTS)} / {len(matrix)}')

## Cell 4 — Stability Check

In [ ]:
from kinetidiff.molecular_dynamics.analysis.trajectory import is_pose_stable

stability_rows = []
threshold = cfg_dict['analysis']['ligand_rmsd_unstable_A']

for (lig_id, receptor, replica_id), res in ANALYSIS_RESULTS.items():
    stable = is_pose_stable(res['rmsd'], threshold_A=threshold)
    stability_rows.append({
        'ligand_id': lig_id, 'receptor': receptor, 'replica_id': replica_id,
        'stable': stable,
    })

stability_df = pd.DataFrame(stability_rows)
print(stability_df.to_string(index=False))
n_unstable = (~stability_df['stable']).sum()
print(f'\n{n_unstable} / {len(stability_df)} runs flagged as potentially unstable (ligand RMSD > {threshold} Å).')

## Cell 5 — MM-GBSA Table & Selectivity

In [ ]:
from kinetidiff.molecular_dynamics.analysis.mmgbsa import mmgbsa_table
from kinetidiff.molecular_dynamics.analysis.selectivity import compute_selectivity

# Build MM-GBSA DataFrame from per-run results
mmgbsa_rows = []
for (lig_id, receptor, replica_id), row in MMGBSA_RESULTS.items():
    mmgbsa_rows.append({
        'ligand_id': lig_id, 'receptor': receptor, 'replica_id': replica_id,
        'vina_score': cfg_dict['campaign']['ligands'][lig_id]['vina'],
        'mmgbsa_mean_kcal': row.get('mmgbsa_mean_kcal', float('nan')),
        'mmgbsa_std_kcal':  row.get('mmgbsa_std_kcal',  float('nan')),
        'mmgbsa_mean_rel':  row.get('mmgbsa_mean_rel',  float('nan')),
        'n_frames':         row.get('n_frames', 0),
        'status':           row.get('status', 'unknown'),
    })

mmgbsa_df = pd.DataFrame(mmgbsa_rows) if mmgbsa_rows else pd.DataFrame()
if not mmgbsa_df.empty:
    display(mmgbsa_df.sort_values(['ligand_id', 'receptor', 'replica_id']))

selectivity_df = compute_selectivity(mmgbsa_df, threshold_kcal=cfg_dict['analysis']['selectivity_threshold_kcal'])
print('\nSelectivity (ΔΔG_bind = ΔG_R206H − ΔG_WT):')
if not selectivity_df.empty:
    display(selectivity_df)
    selectivity_df.to_csv(ANALYSIS_DIR / 'selectivity_ddg.csv', index=False)
else:
    print('  (no data — run analysis array first)')

## Cell 6 — Publication Figures

In [ ]:
from kinetidiff.molecular_dynamics.viz.figures import (
    plot_rmsd_summary, plot_rmsf_comparison,
    plot_mmgbsa_summary, plot_selectivity_scatter, plot_vina_vs_mmgbsa,
)

ligand_ids = cfg_dict['campaign']['priority_order']

fig_rmsd  = plot_rmsd_summary(ANALYSIS_RESULTS, ligand_ids, FIGURES_DIR)
fig_rmsf  = plot_rmsf_comparison(ANALYSIS_RESULTS, ligand_ids, FIGURES_DIR)
fig_mmgbsa = plot_mmgbsa_summary(mmgbsa_df, FIGURES_DIR) if not mmgbsa_df.empty else None
fig_sel    = plot_selectivity_scatter(selectivity_df, FIGURES_DIR) if not selectivity_df.empty else None
fig_vina   = plot_vina_vs_mmgbsa(mmgbsa_df, FIGURES_DIR) if not mmgbsa_df.empty else None

print('Figures written to:', FIGURES_DIR)

## Cell 7 — Master Results CSV

In [ ]:
import numpy as np
from datetime import datetime

master_rows = []
for (lig_id, receptor, replica_id), res in ANALYSIS_RESULTS.items():
    meta = cfg_dict['campaign']['ligands'].get(lig_id, {})
    rmsd_df  = res.get('rmsd', pd.DataFrame())
    hbond_df = res.get('hbonds', pd.DataFrame())
    pocket   = res.get('pocket', {})
    mmgbsa   = MMGBSA_RESULTS.get((lig_id, receptor, replica_id), {})

    master_rows.append({
        'ligand_id':             lig_id,
        'receptor':              receptor,
        'replica_id':            replica_id,
        'vina_score':            meta.get('vina', float('nan')),
        'pkd':                   meta.get('pkd',  float('nan')),
        'qed':                   meta.get('qed',  float('nan')),
        'sa':                    meta.get('sa',   float('nan')),
        'mw_da':                 meta.get('mw',   float('nan')),
        'backbone_rmsd_mean_A':  rmsd_df['backbone_rmsd_A'].mean() if 'backbone_rmsd_A' in rmsd_df.columns else float('nan'),
        'backbone_rmsd_std_A':   rmsd_df['backbone_rmsd_A'].std()  if 'backbone_rmsd_A' in rmsd_df.columns else float('nan'),
        'ligand_rmsd_mean_A':    rmsd_df['ligand_rmsd_A'].mean()   if 'ligand_rmsd_A' in rmsd_df.columns else float('nan'),
        'n_hbonds_persistent':   len(hbond_df) if not hbond_df.empty else 0,
        'top_hbond_occupancy':   hbond_df['occupancy_pct'].max() if not hbond_df.empty and 'occupancy_pct' in hbond_df.columns else float('nan'),
        'pocket_bound_pct':      pocket.get('bound_pct', float('nan')),
        'mmgbsa_mean_kcal':      mmgbsa.get('mmgbsa_mean_kcal', float('nan')),
        'mmgbsa_std_kcal':       mmgbsa.get('mmgbsa_std_kcal',  float('nan')),
        'run_date':              datetime.utcnow().isoformat()[:10],
    })

master_df = pd.DataFrame(master_rows)
master_csv = SCRATCH_DIR / 'master_results.csv'
master_df.to_csv(master_csv, index=False)
print(f'Master CSV: {master_csv}')
if not master_df.empty:
    display(master_df.sort_values(['ligand_id', 'receptor', 'replica_id']))

## Cell 8 — Build Drive Bundle

Strips water, strides by 10×, and writes a manifest.  
Then run `sync_to_drive_bundle.sh` on the login node to `tar` and upload manually.

In [ ]:
BUNDLE_DIR = Path.home() / 'kinetidiff-md-bundle'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Bundle directory: {BUNDLE_DIR}')
print()
print('To build the full bundle, run on the ORCD login node:')
print('  bash scripts/md/sync_to_drive_bundle.sh')
print()
print('Bundle contents will include:')
print('  - trajectories_stripped/   (water-stripped, stride 10×)')
print('  - figures/                 (PNG + SVG)')
print('  - analysis_csvs/           (RMSD, RMSF, H-bonds, MM-GBSA)')
print('  - master_results.csv')
print('  - bundle_manifest.json')
print()
print('Upload to Drive:')
print('  scp -r aaru0302@engaging-login.mit.edu:~/kinetidiff-md-bundle/ .')
print('  # Then drag kinetidiff-md-bundle/ into Google Drive manually.')